# 00 — Constants and geometry check

Prints every constant in `config.py` and recomputes the two that came from
external tools so they can be checked against the paper:

* the solar **P-angle** on 2024‑04‑08 (SunPy) — paper: −26.24°;
* the Sun's and Moon's angular radii and the implied **plate scale** — paper: 1.49″/px.

Celestial North (168° CCW from +x) was measured manually in Stellarium/Affinity
Photo (paper Fig. 5) and is not recomputed here.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
for k in sorted(vars(config)):
    if k.isupper():
        v = getattr(config, k)
        print(f"{k:32s} = {v}")

In [ ]:
# Solar P-angle with SunPy (the legacy value -26.24 deg was computed the same way)
from astropy.time import Time
from sunpy.coordinates import sun

t_mid = Time(config.TOTALITY_MID_UTC)
P = sun.P(t_mid).to("deg").value
B0 = sun.B0(t_mid).to("deg").value
R_sun_arcsec = sun.angular_radius(t_mid).to("arcsec").value
print(f"P-angle at {t_mid.isot}: {P:+.3f} deg   (config: {config.SOLAR_P_ANGLE_DEG:+.2f})")
print(f"B0            : {B0:+.3f} deg")
print(f"Sun angular radius: {R_sun_arcsec:.2f} arcsec")
assert abs(P - config.SOLAR_P_ANGLE_DEG) < 0.05, "P-angle drifted from the paper value"

In [ ]:
# Plate scale implied by the CHT lunar radius and the Skyfield lunar angular radius
eph = utils.Ephemeris()
moon_r_as, sun_r_as, sep_as, pa = eph.sun_moon_geometry(config.TOTALITY_MID_UTC)
leg = pd.read_csv(config.LEGACY_SUN_MOON_CENTERS_CSV)
leg = leg[~leg.filename.str.contains("Position4")]
scale = moon_r_as / leg.moon_radius.mean()
print(f"Moon angular radius {moon_r_as:.2f}\", Sun {sun_r_as:.2f}\", separation {sep_as:.1f}\", PA {pa:.2f} deg")
print(f"mean CHT lunar radius {leg.moon_radius.mean():.2f} px  ->  plate scale {scale:.3f} arcsec/px  (config: {config.PLATE_SCALE_ARCSEC_PER_PIX})")
print(f"Sun radius in pixels at that scale: {sun_r_as/scale:.1f} px  (legacy table mean: {leg.sun_radius.mean():.1f})")